# analysis.ipynb — Deep Message Analysis

Requires: `stemeKit.py`, `blocKit.py`, `analysis_data.py` in the same folder.

In [ ]:
import sqlite3, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from pathlib import Path
from scipy.stats import gaussian_kde

warnings.filterwarnings("ignore", message="Glyph.*missing from font")

import matplotlib.font_manager as fm
cjk_candidates = ["Microsoft JhengHei", "Microsoft YaHei", "Noto Sans CJK TC", "Arial Unicode MS"]
cjk_fonts = [f for f in cjk_candidates if any(f.lower() in x.name.lower() for x in fm.fontManager.ttflist)]
if not cjk_fonts: cjk_fonts = ["DejaVu Sans"]
plt.rcParams["font.sans-serif"] = [cjk_fonts[0], "DejaVu Sans", "Segoe UI Emoji"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96
plt.rcParams["savefig.dpi"] = 96

DB_PATH = str(Path(".") / "messages.db")
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

try:
    from contacts_data import YOUR_ALIASES
    YOUR_NAMES = [name for name, _ in YOUR_ALIASES]
except ImportError:
    YOUR_NAMES = ["Your Name"]
YOU = ", ".join(f"'{n}'" for n in YOUR_NAMES)
print(f"YOU = {YOUR_NAMES}")

from sentence_transformers import SentenceTransformer
_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model ready.")

## 1. Wrapped Stats

In [ ]:
stats = {}
r = conn.execute("SELECT COUNT(*) n, SUM(word_count) w FROM messages WHERE content_type='text'").fetchone()
stats["Total messages"] = f"{r['n']:,}"
stats["Total words"]    = f"{r['w']:,}"
r2 = conn.execute(f"SELECT COUNT(*) n FROM messages WHERE sender IN ({YOU}) AND content_type='text'").fetchone()
stats["Your messages"]  = f"{r2['n']:,}  ({r2['n']/r['n']*100:.1f}%)"
bd = conn.execute("SELECT DATE(timestamp_ms/1000,'unixepoch') d, COUNT(*) n FROM messages GROUP BY d ORDER BY n DESC LIMIT 1").fetchone()
stats["Busiest day"]    = f"{bd['d']}  ({bd['n']:,} messages)"
tc = conn.execute(f"SELECT t.thread_name, COUNT(*) n FROM messages m JOIN threads t ON m.thread_id=t.thread_id WHERE t.is_group=0 GROUP BY t.thread_name ORDER BY n DESC LIMIT 1").fetchone()
stats["Top DM contact"] = f"{tc['thread_name']}  ({tc['n']:,} messages)"
days = pd.read_sql("SELECT DISTINCT DATE(timestamp_ms/1000,'unixepoch') d FROM messages ORDER BY d", conn)["d"].tolist()
max_streak = streak = 1
for i in range(1, len(days)):
    streak = streak+1 if (pd.Timestamp(days[i])-pd.Timestamp(days[i-1])).days==1 else 1
    max_streak = max(max_streak, streak)
stats["Longest streak"] = f"{max_streak} days"
cr = conn.execute("SELECT COUNT(*) n, SUM(call_duration) total_s, MAX(call_duration) max_s FROM messages WHERE content_type='call' AND call_duration IS NOT NULL").fetchone()
if cr["n"]:
    stats["Total calls"]     = f"{cr['n']:,}"
    stats["Total call time"] = f"{cr['total_s']/3600:.1f} hrs"
    stats["Longest call"]    = f"{cr['max_s']//3600}h {(cr['max_s']%3600)//60}m"
yr = conn.execute("SELECT MIN(CAST(STRFTIME('%Y',timestamp_ms/1000,'unixepoch') AS INT)) y0, MAX(CAST(STRFTIME('%Y',timestamp_ms/1000,'unixepoch') AS INT)) y1 FROM messages").fetchone()
stats["Years active"]   = f"{yr['y0']} - {yr['y1']}  ({yr['y1']-yr['y0']+1} yrs)"
fig, ax = plt.subplots(figsize=(10, len(stats)*0.55+0.5))
ax.axis("off")
for i,(k,v) in enumerate(stats.items()):
    y = 1-(i+0.5)/len(stats)
    ax.text(0.02, y, k, ha="left",  va="center", fontsize=12, color="#888")
    ax.text(0.98, y, v, ha="right", va="center", fontsize=13, fontweight="bold", color="#222")
ax.set_title("Your Messaging Wrapped", fontsize=16, fontweight="bold", pad=10)
plt.tight_layout(); plt.show()

## 2. Communication Style per Contact

In [ ]:
TOP_STYLE = 15
EMOJI_RE = re.compile(r"[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F900-\U0001F9FF]", re.UNICODE)
top_contacts = pd.read_sql(
    f"SELECT t.thread_name FROM messages m JOIN threads t ON m.thread_id=t.thread_id "
    f"WHERE t.is_group=0 AND m.sender IN ({YOU}) AND m.content_type='text' "
    f"GROUP BY t.thread_name ORDER BY COUNT(*) DESC LIMIT {TOP_STYLE}", conn
)["thread_name"].tolist()

style_rows = []
for contact in top_contacts:
    df_c = pd.read_sql(
        f"SELECT m.content, m.word_count FROM messages m JOIN threads t ON m.thread_id=t.thread_id "
        f"WHERE t.thread_name=? AND m.sender IN ({YOU}) AND m.content_type='text'",
        conn, params=[contact])
    texts = df_c["content"].dropna().tolist()
    if not texts: continue
    words_all = " ".join(texts).lower().split()
    style_rows.append({
        "contact":       contact,
        "vocab_richness": len(set(words_all))/max(len(words_all),1)*100,
        "question_rate":  sum(1 for t in texts if "?" in t)/len(texts)*100,
        "emoji_rate":     sum(len(EMOJI_RE.findall(t)) for t in texts)/len(texts)*10,
        "avg_msg_len":    df_c["word_count"].mean(),
    })
df_style = pd.DataFrame(style_rows).set_index("contact")

fig, axes = plt.subplots(1, 4, figsize=(18, max(4, len(df_style)*0.4)))
cols   = ["vocab_richness","question_rate","emoji_rate","avg_msg_len"]
titles = ["Vocab richness (%)","Question rate (%)","Emoji rate (x10/msg)","Avg msg length (words)"]
colors = ["#4a90d9","#e07b4a","#f7c948","#74c476"]
for ax,col,title,color in zip(axes,cols,titles,colors):
    d = df_style[col].sort_values(ascending=False)
    ax.barh(d.index, d.values, color=color, alpha=0.85)
    ax.set_title(title, fontsize=9); ax.invert_yaxis()
    ax.tick_params(axis='y', labelsize=8)
plt.suptitle("How you write — per contact", fontsize=13)
plt.tight_layout(); plt.show()

## 3. Social Mask Map

**Axes are pre-defined semantic dimensions (not UMAP coordinates):**
- **X — Energy**: emoji rate, exclamation rate, caps ratio, reaction rate vs. avg message length
- **Y — Intent**: personal pronouns, short questions, sentiment strength, hedging vs. long questions, function words, bare URLs

Features extracted per thread → z-score normalised → composite axis scores → KMeans(6) clusters in 2D.

In [ ]:
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as _VaderSA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, QuantileTransformer

try:
    from contacts_data import KNOWN_CONTACTS
    # reverse map: thread_name → canonical name
    _ALIAS_MAP = {alias: canonical
                  for canonical, aliases in KNOWN_CONTACTS.items()
                  for alias, _platform in aliases}
except ImportError:
    _ALIAS_MAP = {}

_vader_md  = _VaderSA()
_EMOJI_RE  = re.compile(r"[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F900-\U0001F9FF]", re.UNICODE)
_URL_RE    = re.compile(r"https?://\S+|www\.\S+")
_PRON_RE   = re.compile(r"\b(i|me|my|mine|we|us|our|you|your|你|我|妳|咱|我們|你們)\b", re.IGNORECASE)
_HEDGE_RE  = re.compile(r"\b(maybe|perhaps|probably|might|i think|i guess|not sure|idk|dunno|感覺|好像|應該|大概|可能)\b", re.IGNORECASE)
_FUNC_RE   = re.compile(r"\b(the|a|an|is|are|was|were|have|has|had|will|would|could|should|do|does|did|been|being)\b", re.IGNORECASE)

N_MASKS  = 6
MIN_MSGS = 5
SAMPLE   = 300

ENERGY_W = {"emoji_rate":+1, "excl_rate":+1, "caps_ratio":+1, "reaction_rate":+1, "avg_len":-1}
INTENT_W = {"short_q_rate":+1, "pronoun_rate":+1, "sent_strength":+1,
            "hedge_rate":+1,  "url_emoji_rate":+1,
            "long_q_rate":-1, "func_rate":-1,     "bare_url_rate":-1}
_ALL_FEAT_COLS = list(dict.fromkeys(list(ENERGY_W) + list(INTENT_W)))

def _extract_features(texts):
    n  = len(texts)
    wc = [len(t.split()) for t in texts]
    q  = [t for t in texts if "?" in t or "？" in t]
    ql = [len(t.split()) for t in q]
    u  = [t for t in texts if _URL_RE.search(t)]
    vs = [_vader_md.polarity_scores(t)["compound"] for t in texts[:150]]
    return {
        "emoji_rate":     sum(len(_EMOJI_RE.findall(t)) for t in texts) / n,
        "excl_rate":      sum(t.count("!") for t in texts) / n,
        "caps_ratio":     sum(sum(c.isupper() for c in t) / max(len(t), 1) for t in texts) / n,
        "avg_len":        float(np.mean(wc)),
        "reaction_rate":  sum(1 for w in wc if w <= 3) / n,
        "short_q_rate":   sum(1 for l in ql if l <= 5) / n,
        "long_q_rate":    sum(1 for l in ql if l >  5) / n,
        "pronoun_rate":   sum(len(_PRON_RE.findall(t)) for t in texts) / max(sum(wc), 1),
        "hedge_rate":     sum(1 for t in texts if _HEDGE_RE.search(t)) / n,
        "func_rate":      sum(len(_FUNC_RE.findall(t)) for t in texts) / max(sum(wc), 1),
        "url_emoji_rate": sum(1 for t in u if _EMOJI_RE.search(t)) / n,
        "bare_url_rate":  sum(1 for t in u if not _EMOJI_RE.search(t)) / n,
        "sent_strength":  float(np.mean(np.abs(vs))),
    }

def _raw_composite(df_f, scaler):
    Z = pd.DataFrame(scaler.transform(df_f[_ALL_FEAT_COLS]), columns=_ALL_FEAT_COLS)
    e = sum(Z[f] * s for f, s in ENERGY_W.items())
    i = sum(Z[f] * s for f, s in INTENT_W.items())
    return e.values, i.values

def _axis_scores(df_f, scaler, qt_e, qt_i):
    """Quantile-normalised scores mapped to [-1, 1]: uniform spread, sign preserved."""
    e_raw, i_raw = _raw_composite(df_f, scaler)
    e_q = qt_e.transform(e_raw.reshape(-1, 1)).ravel() * 2 - 1
    i_q = qt_i.transform(i_raw.reshape(-1, 1)).ravel() * 2 - 1
    return e_q, i_q

# ── Per-thread message fetch, grouped by canonical identity ──────────────────
threads = pd.read_sql("SELECT thread_id, thread_name, is_group FROM threads", conn)

# Group DM thread_ids under canonical name; unknown names keep their thread_name
identity_groups = {}   # canonical_label → {"is_group": bool, "thread_ids": [...]}
for _, row in threads.iterrows():
    if row["is_group"]:
        label = row["thread_name"]   # group chats stay as-is
    else:
        label = _ALIAS_MAP.get(row["thread_name"], row["thread_name"])
    if label not in identity_groups:
        identity_groups[label] = {"is_group": int(row["is_group"]), "thread_ids": []}
    identity_groups[label]["thread_ids"].append(row["thread_id"])

feat_rows = []
for label, meta in identity_groups.items():
    placeholders = ",".join("?" * len(meta["thread_ids"]))
    df_t = pd.read_sql(
        f"SELECT content FROM messages WHERE thread_id IN ({placeholders}) AND sender IN ({YOU}) "
        f"AND content_type='text' AND LENGTH(content)>5 ORDER BY RANDOM() LIMIT {SAMPLE}",
        conn, params=meta["thread_ids"]
    )
    if len(df_t) < MIN_MSGS:
        continue
    f = _extract_features(df_t["content"].tolist())
    f.update({"thread_name": label, "is_group": meta["is_group"], "n_msgs": len(df_t)})
    feat_rows.append(f)

df_feats = pd.DataFrame(feat_rows).reset_index(drop=True)
print(f"Threads: {len(df_feats)}  ({(df_feats.is_group==0).sum()} DMs, {(df_feats.is_group==1).sum()} GCs)")

# ── Fit scalers ────────────────────────────────────────────────────────────────
_scaler = StandardScaler().fit(df_feats[_ALL_FEAT_COLS])
_e_raw, _i_raw = _raw_composite(df_feats, _scaler)
_qt_e = QuantileTransformer(output_distribution="uniform", random_state=42).fit(_e_raw.reshape(-1, 1))
_qt_i = QuantileTransformer(output_distribution="uniform", random_state=42).fit(_i_raw.reshape(-1, 1))

df_feats["energy"], df_feats["intent"] = _axis_scores(df_feats, _scaler, _qt_e, _qt_i)

# ── KMeans in the 2D quadrant space ───────────────────────────────────────────
_kmeans_2d = KMeans(n_clusters=N_MASKS, random_state=42, n_init=10)
df_feats["mask"] = _kmeans_2d.fit_predict(df_feats[["energy","intent"]].values)

MASK_COLORS = plt.cm.Set2(np.linspace(0, 1, N_MASKS))
cluster_ids = list(range(N_MASKS))

print("\n── Mask Clusters ─────────────────────────────────────────────────────────")
for ci in sorted(cluster_ids, key=lambda c: _kmeans_2d.cluster_centers_[c, 1], reverse=True):
    cx, cy = _kmeans_2d.cluster_centers_[ci]
    e_lbl = "high-energy" if cx > 0 else "restrained"
    i_lbl = "connect"     if cy > 0 else "content/task"
    sub = df_feats[df_feats["mask"] == ci].sort_values("n_msgs", ascending=False)
    print(f"\n[Mask {ci}]  {e_lbl} × {i_lbl}   n={len(sub)}")
    for _, r in sub.head(5).iterrows():
        tag = " [GC]" if r["is_group"] else ""
        print(f"    {r['thread_name']}{tag}   (e={r['energy']:.2f}, i={r['intent']:.2f})")

In [ ]:
# ── EDIT THIS after reading the representatives above ─────────────────────────
# Map cluster index → your human-readable mask label
MASK_NAMES = {
    # 0: "Leisure / BS",
    # 1: "Emotional / Support",
    # 2: "Planning / Logistics",
    # 3: "Hype / Reactions",
    # ...
}
# ──────────────────────────────────────────────────────────────────────────────

mask_labels = [MASK_NAMES.get(i, f"Mask {i}") for i in cluster_ids]
n_masks = len(mask_labels)
MASK_COLORS = plt.cm.Set2(np.linspace(0, 1, n_masks))

print("Current mask labels:")
for i, lbl in enumerate(mask_labels):
    print(f"  {i}: {lbl}")

In [ ]:
mask_labels = [MASK_NAMES.get(i, f"Mask {i}") for i in cluster_ids]

fig, ax = plt.subplots(figsize=(13, 10))
ax.axhline(0, color="#ddd", lw=0.8, zorder=0)
ax.axvline(0, color="#ddd", lw=0.8, zorder=0)

ax_lim = float(df_feats[["energy","intent"]].abs().values.max()) * 1.1

# Quadrant corner labels
for (sx, sy, lbl) in [(1,1,"high-energy + connect"), (-1,1,"restrained + connect"),
                       (1,-1,"high-energy + task"),  (-1,-1,"restrained + task")]:
    ax.text(ax_lim*sx*0.55, ax_lim*sy*0.95, lbl,
            ha="center", fontsize=7.5, color="#bbb", style="italic")

# KDE basins
for ci, mcolor in enumerate(MASK_COLORS):
    pts = df_feats[df_feats["mask"]==ci][["energy","intent"]].values
    if len(pts) < 4:
        continue
    try:
        kde = gaussian_kde(pts.T, bw_method=0.6)
        margin = ax_lim * 0.12
        xi = np.linspace(-ax_lim-margin, ax_lim+margin, 160)
        yi = np.linspace(-ax_lim-margin, ax_lim+margin, 160)
        Xi, Yi = np.meshgrid(xi, yi)
        Zi = kde(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)
        ax.contourf(Xi, Yi, Zi, levels=[Zi.max()*0.10, Zi.max()], colors=[mcolor], alpha=0.15)
        ax.contour( Xi, Yi, Zi, levels=[Zi.max()*0.20], colors=[mcolor], linewidths=0.9, alpha=0.55)
        pk = np.unravel_index(Zi.argmax(), Zi.shape)
        ax.text(Xi[pk], Yi[pk], mask_labels[ci],
                ha="center", va="center", fontsize=9, color=mcolor, fontweight="bold", alpha=0.8)
    except Exception:
        pass

# Thread dots
for ci, (mlabel, mcolor) in enumerate(zip(mask_labels, MASK_COLORS)):
    for is_grp, mkr in [(0, "o"), (1, "s")]:
        sub = df_feats[(df_feats["mask"]==ci) & (df_feats["is_group"]==is_grp)]
        if sub.empty:
            continue
        sz = np.clip(np.sqrt(sub["n_msgs"].values) * 5, 30, 400)
        ax.scatter(sub["energy"], sub["intent"], c=[mcolor], s=sz, marker=mkr,
                   alpha=0.9, edgecolors="white", linewidths=0.8, zorder=5)

# Labels: top-10 DMs + all GCs
for _, r in df_feats[df_feats["is_group"]==0].nlargest(10, "n_msgs").iterrows():
    ax.annotate(r["thread_name"], (r["energy"], r["intent"]),
                textcoords="offset points", xytext=(4, 3), fontsize=7, color="#223")
for _, r in df_feats[df_feats["is_group"]==1].iterrows():
    ax.annotate(r["thread_name"], (r["energy"], r["intent"]),
                textcoords="offset points", xytext=(4, 3), fontsize=7, color="#a04040")

ax.set_xlabel("Energy  ←  restrained · · · high-engage  →", fontsize=10)
ax.set_ylabel("Intent  ←  content/task · · · connect/social  →", fontsize=10)
ax.set_xlim(-ax_lim-0.5, ax_lim+0.5)
ax.set_ylim(-ax_lim-0.5, ax_lim+0.5)

legend_patches = [mpatches.Patch(color=MASK_COLORS[i], label=mask_labels[i]) for i in cluster_ids]
legend_patches += [plt.scatter([],[], marker="o", c="#999", s=40, label="DM"),
                   plt.scatter([],[], marker="s", c="#999", s=40, label="Group (red label)")]
ax.legend(handles=legend_patches, loc="upper left", bbox_to_anchor=(1, 1), fontsize=8)
ax.set_title("Social Mask Map\nAxes = lexical proxies  ·  Basins = KDE per cluster  ·  size ~ msg count", fontsize=11)
plt.tight_layout(); plt.show()

## 4. Relationship Drift — Trajectory per Contact

Same quadrant space. Track each DM contact's yearly centroid as an arrow path.
Color = dominant mask in that year.

In [ ]:
DRIFT_CONTACTS = 8
DRIFT_PER_YEAR = 200

# Top contacts by merged message count (using canonical identity)
_canon_dm = (
    df_feats[df_feats["is_group"] == 0]
    .sort_values("n_msgs", ascending=False)
    .head(DRIFT_CONTACTS)["thread_name"]
    .tolist()
)

# Build reverse canonical → thread_ids map from identity_groups
_canon_thread_ids = {label: meta["thread_ids"] for label, meta in identity_groups.items()}

PERSON_COLORS = plt.cm.tab10(np.linspace(0, 0.9, len(_canon_dm)))
fig, ax = plt.subplots(figsize=(11, 10))
ax.axhline(0, color="#eee", lw=0.8); ax.axvline(0, color="#eee", lw=0.8)

all_pts_x, all_pts_y = [], []

for contact, pcolor in zip(_canon_dm, PERSON_COLORS):
    tids = _canon_thread_ids.get(contact, [])
    if not tids:
        continue
    placeholders = ",".join("?" * len(tids))
    df_c = pd.read_sql(
        f"SELECT STRFTIME('%Y', m.timestamp_ms/1000,'unixepoch') yr, m.content "
        f"FROM messages m WHERE m.thread_id IN ({placeholders}) "
        f"AND m.sender IN ({YOU}) AND m.content_type='text' AND LENGTH(m.content)>5",
        conn, params=tids
    )
    traj = []
    for yr, grp in df_c.groupby("yr"):
        sample_texts = grp.sample(min(DRIFT_PER_YEAR, len(grp)), random_state=42)["content"].tolist()
        if len(sample_texts) < 5:
            continue
        feat_df = pd.DataFrame([_extract_features(sample_texts)])
        for col in _ALL_FEAT_COLS:
            if col not in feat_df.columns:
                feat_df[col] = 0.0
        e_arr, i_arr = _axis_scores(feat_df, _scaler, _qt_e, _qt_i)
        e_val, i_val = float(e_arr[0]), float(i_arr[0])
        dom = int(_kmeans_2d.predict([[e_val, i_val]])[0])
        traj.append((yr, e_val, i_val, dom))

    if len(traj) < 2:
        continue

    xs = [t[1] for t in traj]; ys = [t[2] for t in traj]
    all_pts_x.extend(xs);       all_pts_y.extend(ys)

    for k in range(len(traj)-1):
        ax.annotate("", xy=(xs[k+1], ys[k+1]), xytext=(xs[k], ys[k]),
                    arrowprops=dict(arrowstyle="->", color=pcolor, lw=1.5))
    for yr, x, y, dom in traj:
        ax.scatter(x, y, color=MASK_COLORS[dom], s=70, zorder=5,
                   edgecolors=pcolor, linewidths=1.8)
        ax.text(x, y, f" {yr}", fontsize=6.5, color=pcolor, va="center")
    ax.scatter(xs[0], ys[0], color=pcolor, s=130, marker="o", zorder=6, label=contact)

person_legend = ax.legend(loc="lower right", fontsize=8, title="Contact")
mask_patches  = [mpatches.Patch(color=MASK_COLORS[i], label=mask_labels[i]) for i in cluster_ids]
ax.add_artist(person_legend)
ax.legend(handles=mask_patches, loc="upper left", bbox_to_anchor=(1, 1), fontsize=8, title="Mask (dot fill)")

if all_pts_x:
    pad = max(max(all_pts_x)-min(all_pts_x), max(all_pts_y)-min(all_pts_y)) * 0.1 + 0.05
    ax.set_xlim(min(all_pts_x)-pad, max(all_pts_x)+pad)
    ax.set_ylim(min(all_pts_y)-pad, max(all_pts_y)+pad)

ax.set_xlabel("Energy  ←  restrained · · · high-engage  →", fontsize=10)
ax.set_ylabel("Intent  ←  content/task · · · connect/social  →", fontsize=10)
ax.set_title("Relationship Drift — Yearly Centroid\nArrow = person  ·  dot fill = dominant mask that year", fontsize=11)
plt.tight_layout(); plt.show()

## 5. Sentiment: You vs Them (VADER)

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()

SENT_CONTACTS = 8
SENT_MSGS = 500

sent_contacts = pd.read_sql(
    f"SELECT t.thread_name FROM messages m JOIN threads t ON m.thread_id=t.thread_id "
    f"WHERE t.is_group=0 AND m.content_type='text' "
    f"GROUP BY t.thread_name ORDER BY COUNT(*) DESC LIMIT {SENT_CONTACTS}",
    conn
)["thread_name"].tolist()

def score_sample(contact, in_you):
    clause = f"IN ({YOU})" if in_you else f"NOT IN ({YOU})"
    df_c = pd.read_sql(
        f"SELECT m.content FROM messages m JOIN threads t ON m.thread_id=t.thread_id "
        f"WHERE t.thread_name=? AND m.sender {clause} "
        f"AND m.content_type='text' AND LENGTH(m.content)>3 ORDER BY RANDOM() LIMIT {SENT_MSGS}",
        conn, params=[contact])
    scores = [vader.polarity_scores(t) for t in df_c["content"].tolist()]
    if not scores: return {"pos":0,"neu":0,"neg":0,"compound":0}
    return {k: np.mean([s[k] for s in scores]) for k in ["pos","neu","neg","compound"]}

rows_you  = [score_sample(c, True)  for c in sent_contacts]
rows_them = [score_sample(c, False) for c in sent_contacts]
df_you  = pd.DataFrame(rows_you,  index=sent_contacts)
df_them = pd.DataFrame(rows_them, index=sent_contacts)

fig, ax2 = plt.subplots(figsize=(9, max(4, len(sent_contacts)*0.5)))
y = np.arange(len(sent_contacts))
w = 0.35
ax2.barh(y-w/2, df_you["compound"],  w, label="You",  color="#4a90d9")
ax2.barh(y+w/2, df_them["compound"], w, label="Them", color="#e07b4a")
ax2.set_yticks(y); ax2.set_yticklabels(sent_contacts, fontsize=9)
ax2.axvline(0, color="#ccc", lw=0.8)
ax2.set_xlabel("Compound sentiment  (-1 = negative, +1 = positive)")
ax2.set_title("Overall Tone: You vs Them (VADER)"); ax2.legend(); ax2.invert_yaxis()
plt.tight_layout(); plt.show()